# ReparoS Scaling Law Benchmark — Pilot Empirical Study (Arms A - F)
### Testing Architectural Scaling vs. Knowledge Distillation on 300,000 Clean Pairs

This notebook systematically trains and benchmarks 6 Transformer architectures on the **100% pure clean Pilot Dataset (300,000 pairs)**.

**Progressive Pipeline Guarantee:**
- Mỗi Arm sau khi train xong sẽ **tự động Convert $\rightarrow$ Đánh giá ngay $\rightarrow$ Cập nhật Bảng So Sánh $\rightarrow$ Lưu file `report.md` vào đĩa**.
- Kể cả khi bạn dừng giữa chừng hoặc chỉ chạy 1 vài Arm, kết quả của các Arm đã chạy đều được bảo toàn 100% trong `/kaggle/working/`!

| Arm | Architecture | Enc | Dec | $d_{model}$ | $d_{ff}$ | Heads | Total Params | Research Hypothesis |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| **A** | **2E1D d128** | 2 | 1 | 128 | 2048 | 8 | 6.47M | **Baseline**: Ultra-low latency production SLA (~5.1ms) |
| **B** | **4E4D d128** | 4 | 4 | 128 | 2048 | 8 | 9.63M | **Reproduce large paper**: Balanced medium-deep model |
| **C** | **6E6D d128** | 6 | 6 | 128 | 2048 | 8 | 12.13M | **Reproduce largest paper**: Vaswani standard depth |
| **D** | **4E1D d128** | 4 | 1 | 128 | 2048 | 8 | 7.65M | **Gain from Encoder?**: Deeper parallel Enc, fast $O(1)$ Dec step |
| **E** | **2E2D d128** | 2 | 2 | 128 | 2048 | 8 | 7.13M | **Decoder worth it?**: Extra autoregressive layer vs latency |
| **F** | **2E1D d256** | 2 | 1 | 256 | 2048 | 8 | 13.44M | **Depth vs Width**: Shallow depth with doubled embedding dimension |


In [ ]:
# Benchmark Hyperparameters
TRAIN_STEPS = 10_000         # 10k steps covers ~15-20 epochs of 300k pairs with 32k tokens/batch
VALID_STEPS = 1_000
SAVE_CHECKPOINT_STEPS = 5_000
KEEP_CHECKPOINTS = 2
BATCH_SIZE_TOKENS = 32_768   # 32,768 tokens/batch (FP16)
BUCKET_SIZE = 65_536
NUM_WORKERS = 4
MODEL_DTYPE = "fp16"
SEED = 2026
REPORT_EVERY = 1_000

# Select which arms to run (Default: all 6 arms, or choose specific arms)
ACTIVE_ARMS = ["arm_a_2e1d_d128", "arm_b_4e4d_d128", "arm_c_6e6d_d128", "arm_d_4e1d_d128", "arm_e_2e2d_d128", "arm_f_2e1d_d256"]


In [ ]:
import json, os, shutil, subprocess, sys, time, argparse
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
REPO_ROOT = Path('/kaggle/working/QU-solution')

# 1. Locate repository and dataset
print("Locating dataset under /kaggle/input...")
extracted_candidate = None
zip_candidate = None

for p in KAGGLE_INPUT.rglob('data/base_v3_pilot'):
    if p.is_dir():
        extracted_candidate = p.parent.parent
        break

if not extracted_candidate:
    zips = sorted(list(KAGGLE_INPUT.rglob('*.zip')))
    if zips:
        zip_candidate = zips[0]

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
REPO_ROOT.mkdir(parents=True, exist_ok=True)

if extracted_candidate:
    print(f"Found extracted dataset at: {extracted_candidate}")
    for folder in ['src', 'scripts', 'data', 'experiments', 'benchmark']:
        src_dir = extracted_candidate / folder
        dst_dir = REPO_ROOT / folder
        if src_dir.exists():
            print(f"  Copying {folder}/ to {dst_dir}...")
            shutil.copytree(src_dir, dst_dir)
elif zip_candidate:
    print(f"Found zip bundle at: {zip_candidate}")
    print("Unpacking zip bundle into /kaggle/working/QU-solution...")
    subprocess.run(['unzip', '-q', str(zip_candidate), '-d', str(REPO_ROOT)], check=True)
else:
    print("WARNING: /kaggle/input dataset not found. Running in local repository mode.")
    REPO_ROOT = Path('.').resolve()

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))

# 2. Dependencies
try:
    import numpy as np
    if int(np.__version__.split('.')[0]) >= 2:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2', '--quiet'], check=True)
except Exception:
    pass

try:
    import onmt
    import sentencepiece
    import ctranslate2
    import torch
    print("Dependencies (OpenNMT-py, sentencepiece, ctranslate2, torch) ready.")
except ImportError:
    print("Installing OpenNMT-py==3.5.1, sentencepiece, ctranslate2, numpy<2...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2', 'OpenNMT-py==3.5.1', 'sentencepiece', 'ctranslate2', '--quiet'], check=True)
    import torch, ctranslate2

# Fix PyTorch 2.6 safe globals unpickling for OpenNMT checkpoints
try:
    torch.serialization.add_safe_globals([argparse.Namespace])
except Exception:
    pass


In [ ]:
DATA_DIR = REPO_ROOT / 'data/base_v3_pilot'
TOKENIZER_MODEL = REPO_ROOT / 'data/tokenizer_v3/tokenizer.model'
EVAL_DIR = REPO_ROOT / 'data/base_v3_eval'

assert (DATA_DIR / 'train.src').exists(), f"train.src not found in {DATA_DIR}!"
assert (DATA_DIR / 'train.tgt').exists(), f"train.tgt not found in {DATA_DIR}!"
assert TOKENIZER_MODEL.exists(), f"tokenizer.model not found in {TOKENIZER_MODEL}!"

src_lines = sum(1 for _ in open(DATA_DIR / 'train.src', 'r', encoding='utf-8'))
tgt_lines = sum(1 for _ in open(DATA_DIR / 'train.tgt', 'r', encoding='utf-8'))
print(f"Verified Pilot Dataset: {src_lines:,} src lines | {tgt_lines:,} tgt lines")
assert src_lines == tgt_lines == 300_000, f"Expected 300,000 pairs, found {src_lines:,}"

# Build Shared Vocab once for all arms
vocab_src = DATA_DIR / 'vocab.src'
vocab_tgt = DATA_DIR / 'vocab.tgt'
config_sample = REPO_ROOT / 'experiments/pilot_scaling/configs/arm_a_2e1d_d128.json'

if not vocab_src.is_file() or not vocab_tgt.is_file() or vocab_src.stat().st_size == 0:
    print("Building OpenNMT vocab with Tokenizer V3 (SPM 12k)...")
    subprocess.run([
        sys.executable, '-m', 'onmt.bin.build_vocab',
        '-config', str(config_sample), '-n_sample', '-1'
    ], check=True)
    print("Shared vocab built successfully!")
else:
    print("Shared vocab already present. Proceeding.")


In [ ]:
import sentencepiece as spm
import ctranslate2

sp = spm.SentencePieceProcessor()
sp.load(str(TOKENIZER_MODEL))

SUITES = {
    "plasticity": (EVAL_DIR / 'plasticity.src', EVAL_DIR / 'plasticity.tgt'),
    "retention": (EVAL_DIR / 'retention.src', EVAL_DIR / 'retention.tgt'),
    "user_centric": (EVAL_DIR / 'user_centric.src', EVAL_DIR / 'user_centric.tgt'),
    "protection_seen": (EVAL_DIR / 'protection_seen.src', EVAL_DIR / 'protection_seen.tgt'),
    "protection_heldout": (EVAL_DIR / 'protection_heldout.src', EVAL_DIR / 'protection_heldout.tgt')
}

def predict_batch(translator, texts, batch_size=64):
    results = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        tokenized = [sp.encode_as_pieces(t) for t in chunk]
        translations = translator.translate_batch(tokenized, beam_size=1, repetition_penalty=1.2)
        for trans in translations:
            hyp = trans.hypotheses[0] if hasattr(trans, 'hypotheses') else trans[0]['tokens']
            out_text = sp.decode_pieces(hyp)
            results.append(out_text)
    return results

def eval_suite(translator, src_file, tgt_file):
    srcs = [l.strip() for l in open(src_file, 'r', encoding='utf-8').readlines()]
    tgts = [l.strip() for l in open(tgt_file, 'r', encoding='utf-8').readlines()]
    t0 = time.time()
    preds = predict_batch(translator, srcs)
    dur = time.time() - t0
    correct = sum(1 for p, t in zip(preds, tgts) if p.strip().lower() == t.strip().lower())
    acc = correct / len(srcs) * 100
    lat_ms = (dur / len(srcs)) * 1000
    return acc, lat_ms

OUTPUT_REPORT_MD = Path('/kaggle/working/scaling_benchmark_report.md')
OUTPUT_RESULTS_JSON = Path('/kaggle/working/scaling_benchmark_results.json')

def render_and_save_table(eval_matrix, training_results):
    lines = []
    lines.append("# ReparoS Scaling Law Benchmark Results\n")
    lines.append("| Arm | Architecture | Train Time | P50 Latency | Mean Acc | Telex (Plasticity) | OSM (Retention) | User Centric |")
    lines.append("|---|---|---|---|---|---|---|---|")
    for arm in ACTIVE_ARMS:
        if arm not in eval_matrix:
            continue
        m = eval_matrix[arm]
        t_min = training_results.get(arm, {}).get('duration_sec', 0) / 60
        row = f"| **{arm}** | {arm.replace('arm_', '')} | {t_min:.1f}m | {m['mean_lat']:.2f}ms | **{m['mean_acc']:.2f}%** | {m['plasticity']:.1f}% | {m['retention']:.1f}% | {m['user_centric']:.1f}% |"
        lines.append(row)
    table_content = '\n'.join(lines) + '\n'
    
    # Print to cell output
    print("\n" + "="*85)
    print(table_content)
    print("="*85)
    
    # Save permanently to /kaggle/working/
    with open(OUTPUT_REPORT_MD, 'w', encoding='utf-8') as f:
        f.write(table_content)
    with open(OUTPUT_RESULTS_JSON, 'w', encoding='utf-8') as f:
        json.dump({'eval_matrix': eval_matrix, 'training_results': training_results}, f, indent=2)
    print(f"Saved persistent report to {OUTPUT_REPORT_MD} and {OUTPUT_RESULTS_JSON}!")


In [ ]:
checkpoints_root = Path('/kaggle/working/checkpoints/pilot_scaling')
checkpoints_root.mkdir(parents=True, exist_ok=True)

# Load existing results if resuming or rerunning
eval_matrix = {}
training_results = {}
if OUTPUT_RESULTS_JSON.exists():
    try:
        saved = json.load(open(OUTPUT_RESULTS_JSON, 'r', encoding='utf-8'))
        eval_matrix = saved.get('eval_matrix', {})
        training_results = saved.get('training_results', {})
        print(f"Loaded previous progress: {list(eval_matrix.keys())}")
    except Exception:
        pass

for arm in ACTIVE_ARMS:
    print(f"\n{'#'*85}")
    print(f"### PIPELINE: {arm.upper()}")
    print(f"{'#'*85}")
    
    arm_ckpt_dir = checkpoints_root / arm
    arm_ckpt_dir.mkdir(parents=True, exist_ok=True)
    arm_ct2_dir = arm_ckpt_dir / 'ct2_model'
    
    # --- STEP 1: TRAINING ---
    ckpts = sorted(list(arm_ckpt_dir.glob('model_step_*.pt')))
    if ckpts:
        print(f"Found existing checkpoint for {arm}: {ckpts[-1].name}. Skipping retraining.")
        if arm not in training_results:
            training_results[arm] = {"duration_sec": 0}
    else:
        cfg_path = REPO_ROOT / f'experiments/pilot_scaling/configs/{arm}.json'
        with open(cfg_path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
        cfg['save_data'] = str(DATA_DIR / 'vocab')
        cfg['src_vocab'] = str(DATA_DIR / 'vocab.src')
        cfg['tgt_vocab'] = str(DATA_DIR / 'vocab.tgt')
        cfg['data']['corpus_1']['path_src'] = str(DATA_DIR / 'train.src')
        cfg['data']['corpus_1']['path_tgt'] = str(DATA_DIR / 'train.tgt')
        cfg['data']['valid']['path_src'] = str(DATA_DIR / 'valid.src')
        cfg['data']['valid']['path_tgt'] = str(DATA_DIR / 'valid.tgt')
        cfg['src_subword_model'] = str(TOKENIZER_MODEL)
        cfg['tgt_subword_model'] = str(TOKENIZER_MODEL)
        cfg['save_model'] = str(arm_ckpt_dir / 'model')
        cfg['train_steps'] = TRAIN_STEPS
        cfg['valid_steps'] = VALID_STEPS
        cfg['save_checkpoint_steps'] = SAVE_CHECKPOINT_STEPS
        cfg['keep_checkpoint'] = KEEP_CHECKPOINTS
        cfg['batch_size'] = BATCH_SIZE_TOKENS
        cfg['report_every'] = REPORT_EVERY
        
        active_cfg_path = arm_ckpt_dir / 'active_config.json'
        with open(active_cfg_path, 'w', encoding='utf-8') as f:
            json.dump(cfg, f, indent=2)
            
        print(f"[1/3] Training {arm} (Enc: {cfg['enc_layers']}, Dec: {cfg['dec_layers']}, Dim: {cfg['hidden_size']})...")
        log_file = arm_ckpt_dir / 'train.log'
        t0 = time.time()
        with open(log_file, 'w', encoding='utf-8') as log_f:
            proc = subprocess.Popen(
                [sys.executable, '-m', 'onmt.bin.train', '-config', str(active_cfg_path)],
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
            )
            for line in proc.stdout:
                log_f.write(line)
                if 'Step ' in line and ('acc:' in line or 'loss:' in line or 'Saving checkpoint' in line):
                    print(f"  [{time.strftime('%H:%M:%S')}] {line.strip()}", flush=True)
            proc.wait()
        t_dur = time.time() - t0
        if proc.returncode != 0:
            print(f"ERROR in training {arm}! Last lines:")
            print(''.join(open(log_file).readlines()[-20:]))
            raise RuntimeError(f"Training failed for {arm}")
        training_results[arm] = {"duration_sec": t_dur}
        print(f"Training finished in {t_dur/60:.1f} minutes!")
        ckpts = sorted(list(arm_ckpt_dir.glob('model_step_*.pt')))
        
    # --- STEP 2: CTRANSLATE2 CONVERSION (Fail-proof) ---
    latest_ckpt = ckpts[-1]
    if not arm_ct2_dir.exists() or not (arm_ct2_dir / 'model.bin').exists():
        print(f"[2/3] Converting {latest_ckpt.name} to CTranslate2 (FP16)...")
        if arm_ct2_dir.exists():
            shutil.rmtree(arm_ct2_dir)
        try:
            converter = ctranslate2.converters.OpenNMTPyConverter(str(latest_ckpt), unsafe_deserialization=True)
            converter.convert(str(arm_ct2_dir), quantization='float16', force=True)
            print(f"  Converted successfully to {arm_ct2_dir}")
        except Exception as e:
            print(f"  Python API conversion warning: {e}. Fallback to CLI...")
            cmd = [
                'ct2-opennmt-py-converter',
                '--model_path', str(latest_ckpt),
                '--output_dir', str(arm_ct2_dir),
                '--quantization', 'float16',
                '--unsafe_deserialization',
                '--force'
            ]
            subprocess.run(cmd, check=True)
            print(f"  Converted successfully via CLI to {arm_ct2_dir}")
    else:
        print(f"[2/3] CTranslate2 model already present at {arm_ct2_dir}.")
        
    # --- STEP 3: BENCHMARK EVALUATION ---
    if arm not in eval_matrix:
        print(f"[3/3] Evaluating {arm} on 5 Frozen Test Suites (3,150 queries)...")
        translator = ctranslate2.Translator(str(arm_ct2_dir), device='cuda' if ctranslate2.get_cuda_device_count() > 0 else 'cpu')
        arm_metrics = {}
        for s_name, (s_src, s_tgt) in SUITES.items():
            acc, lat = eval_suite(translator, s_src, s_tgt)
            arm_metrics[s_name] = acc
            arm_metrics[f"{s_name}_lat"] = lat
            print(f"  {s_name:<20}: {acc:6.2f}% | Latency: {lat:.2f}ms")
        mean_acc = sum(arm_metrics[s] for s in SUITES) / len(SUITES)
        mean_lat = sum(arm_metrics[f"{s}_lat"] for s in SUITES) / len(SUITES)
        arm_metrics["mean_acc"] = mean_acc
        arm_metrics["mean_lat"] = mean_lat
        eval_matrix[arm] = arm_metrics
        print(f"  >> OVERALL MEAN: {mean_acc:.2f}% | Mean Latency: {mean_lat:.2f}ms")
    else:
        print(f"[3/3] Already evaluated: Mean Acc = {eval_matrix[arm]['mean_acc']:.2f}%")
        
    # --- STEP 4: UPDATE AND PERSIST COMPARISON TABLE IMMEDIATELY ---
    render_and_save_table(eval_matrix, training_results)

print("\nAll active arms processed successfully!")


In [ ]:
print("\n" + "="*90)
print("### FINAL ARCHITECTURAL SCALING & DISTILLATION VERDICT")
print("="*90)

if 'arm_a_2e1d_d128' in eval_matrix and len(eval_matrix) > 1:
    baseline_acc = eval_matrix['arm_a_2e1d_d128']['mean_acc']
    best_arm = max(eval_matrix.keys(), key=lambda a: eval_matrix[a]['mean_acc'])
    best_acc = eval_matrix[best_arm]['mean_acc']
    best_lat = eval_matrix[best_arm]['mean_lat']
    gain = best_acc - baseline_acc
    
    print(f"Baseline Arm A (2E1D): {baseline_acc:.2f}%")
    print(f"Best Arm: {best_arm} ({best_acc:.2f}%, Gain vs Baseline: +{gain:.2f}%)")
    
    if gain >= 5.0 and best_lat > 5.5:
        print("\n==> DECISION: ADOPT KNOWLEDGE DISTILLATION!")
        print(f"    Deep model {best_arm} provides significant accuracy gain (+{gain:.2f}%), but exceeds 5.5ms latency SLA ({best_lat:.2f}ms).")
        print(f"    RECOMMENDATION: Train {best_arm} as Teacher on full 4M clean data, then distill into Student Arm A / D.")
    elif 'arm_d_4e1d_d128' in eval_matrix and eval_matrix['arm_d_4e1d_d128']['mean_acc'] >= (best_acc - 1.0) and eval_matrix['arm_d_4e1d_d128']['mean_lat'] <= 5.5:
        print("\n==> DECISION: ADOPT ASYMMETRIC ARM D (4E1D d128) DIRECTLY!")
        print("    Arm D captures virtually all the depth gain while keeping latency within 5.5ms SLA.")
        print("    RECOMMENDATION: Train Arm D directly for production without distillation complexity!")
    else:
        print("\n==> DECISION: NO DISTILLATION NEEDED!")
        print("    Scaling depth/width provides marginal gain (< 2%). Baseline 2E1D capacity is already sufficient.")
        print("    RECOMMENDATION: Keep Arm A (2E1D), prioritize data diversity and coverage.")
else:
    print(f"Only {len(eval_matrix)} arm(s) evaluated so far. Run more arms to compare scaling laws.")
print("="*90)
